In [ ]:
from datasets import load_dataset

dataset = load_dataset("tcltcl/small-simple-wikipedia", split="train")
texts = dataset["text"]

chunk_size = 200
overlap = 50

chunks = []

for text in texts:
    words = text.split()
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)


In [ ]:
from sentence_transformers import SentenceTransformer

retriever_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = retriever_model.encode(chunks)

In [ ]:
import numpy as np

def retrieve_chunk(question):
    question_embedding = retriever_model.encode([question])
    
    similarities = retriever_model.similarity(question_embedding, embeddings)
    
    best_idx = np.argmax(similarities)
    return chunks[best_idx]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceH4/zephyr-7b-beta",
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_answer(question):
    context = retrieve_chunk(question)
    prompt = f"<|user|>\nUse the following context to answer the question.\n\nContext: {context}\n\nQuestion: {question}\n<|assistant|>"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs)

    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = answer.split("<|assistant|>")[-1].strip()
    return answer

In [ ]:
print(generate_answer("What month is after March?"))
print(generate_answer("Branch of astronomy that deals with the origin, structure, evolution and space-time relationships of the universe"))
print(generate_answer("Dairy product that is made from milk"))